# 01 - EDA inicial: Cardiovascular Disease Dataset

Este notebook realiza la primera etapa del **Análisis Exploratorio de Datos (EDA)**.

### Objetivos
1. Cargar correctamente el dataset.
2. Comprender su estructura.
3. Evaluar calidad de los datos.
4. Analizar la variable objetivo `cardio`.
5. Convertir la edad de días a años.
6. Detectar posibles valores anómalos **sin eliminarlos todavía**.

> En esta etapa hacemos diagnóstico. La limpieza y el preprocesamiento se realizarán posteriormente.

In [ ]:
from pathlib import Path
import pandas as pd

DATA_FILE = Path('cardio_train.csv')

if not DATA_FILE.exists():
    raise FileNotFoundError("Coloca 'cardio_train.csv' en la misma carpeta del notebook.")

df = pd.read_csv(DATA_FILE, sep=';')
df.head()

## 1. Dimensiones y estructura general

In [ ]:
print(f'Filas: {df.shape[0]:,}')
print(f'Columnas: {df.shape[1]}')
print('Columnas:', df.columns.tolist())

In [ ]:
df.info()

## 2. Estadística descriptiva

In [ ]:
df.describe().T

## 3. Valores faltantes y duplicados

In [ ]:
missing = df.isna().sum()
print(missing)
print('\nTotal de valores faltantes:', int(missing.sum()))
print('Filas duplicadas:', df.duplicated().sum())

## 4. Número de valores únicos por variable

In [ ]:
df.nunique()

## 5. Variable objetivo `cardio`

In [ ]:
cardio_counts = df['cardio'].value_counts().sort_index()
cardio_pct = df['cardio'].value_counts(normalize=True).sort_index() * 100

pd.DataFrame({
    'frecuencia': cardio_counts,
    'porcentaje': cardio_pct.round(2)
})

## 6. Conversión de edad

En este dataset, `age` está expresada en **días**. Para facilitar la interpretación se crea `age_years`.

In [ ]:
df['age_years'] = df['age'] / 365.25
df['age_years'].describe().round(2)

## 7. Distribución de variables categóricas y binarias

In [ ]:
categorical_cols = ['gender', 'cholesterol', 'gluc', 'smoke', 'alco', 'active', 'cardio']

for col in categorical_cols:
    print(f'\n--- {col} ---')
    counts = df[col].value_counts().sort_index()
    pct = df[col].value_counts(normalize=True).sort_index() * 100
    display(pd.DataFrame({'frecuencia': counts, 'porcentaje': pct.round(2)}))

## 8. Detección inicial de posibles valores anómalos

Aquí **no eliminamos datos**. Solo cuantificamos valores que merecen revisión.

In [ ]:
checks = {
    'height < 120 cm': (df['height'] < 120).sum(),
    'height > 220 cm': (df['height'] > 220).sum(),
    'weight < 30 kg': (df['weight'] < 30).sum(),
    'weight > 200 kg': (df['weight'] > 200).sum(),
    'ap_hi <= 0': (df['ap_hi'] <= 0).sum(),
    'ap_hi > 250': (df['ap_hi'] > 250).sum(),
    'ap_lo <= 0': (df['ap_lo'] <= 0).sum(),
    'ap_lo > 150': (df['ap_lo'] > 150).sum(),
    'ap_hi < ap_lo': (df['ap_hi'] < df['ap_lo']).sum(),
}

pd.Series(checks, name='cantidad')

## 9. Creación exploratoria del IMC

La fórmula es:

$$BMI = \frac{peso\;(kg)}{altura\;(m)^2}$$

> Como todavía existen posibles alturas/pesos erróneos, este IMC es únicamente exploratorio.

In [ ]:
df['bmi'] = df['weight'] / ((df['height'] / 100) ** 2)
df['bmi'].describe().round(2)

## 10. Comparación preliminar respecto a `cardio`

In [ ]:
numeric_compare = ['age_years', 'height', 'weight', 'ap_hi', 'ap_lo', 'bmi']
df.groupby('cardio')[numeric_compare].mean().round(2)

In [ ]:
pd.crosstab(df['cardio'], df['cholesterol'], normalize='index').mul(100).round(2)

In [ ]:
pd.crosstab(df['cardio'], df['gluc'], normalize='index').mul(100).round(2)

## 11. Conclusiones de esta primera etapa

- El dataset debe cargarse usando `sep=';'`.
- Se revisaron dimensiones, tipos de datos, faltantes y duplicados.
- `cardio` define un problema de clasificación binaria.
- `age` está expresada en días y se creó `age_years` para interpretación.
- Existen posibles valores anómalos en presión arterial, altura y peso.
- Se creó `bmi` solo con fines exploratorios.
- Todavía **no se eliminó ni corrigió ningún registro**.

### Siguiente paso recomendado
Realizar el **EDA univariado gráfico** y luego el **EDA bivariado respecto a `cardio`**.